<a href="https://colab.research.google.com/github/muaaz-tahir/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muaaz-tahir/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [10]:
%pip -q install duckdb

import duckdb
con = duckdb.connect()

# Paste your HF token here using Colab's Secrets panel (key icon on the left sidebar),
# name it HF_TOKEN, then run this:
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")

REL = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"

# Let's see the real column names before we write anything else
con.sql(f"DESCRIBE SELECT * FROM {REL}").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

1. One row = one page, for one client, on one specific day (report_date).

2. Table used: fact_content_daily_performance, filtered to one month
(month=2026-03), a mid-panel month, not the final month.

3. Time window: March 2026 only, for developing my logic. The final month
(June 2026) stays sealed as a test month, not touched during development.

4. What I would predict: same proxy idea as before, whether a page is
declining while still getting real search demand, but now built day by day
instead of as one 90-day total.

5. What I deliberately exclude: any FlyRank product decision flag (like
health_score or priority_score), since those already encode a human
decision and would make my result circular, not a real finding.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature (safe to use): gsc_impressions, gsc_clicks, gsc_avg_position,
ga4_sessions, scroll_events. All of these are simply recorded on the day
they happened, so they are known at decision time.

Label/proxy: a column I build myself, showing whether a page's traffic
dropped sharply the next day. Never a feature, since it uses future
information.

Context (for joining/grouping only, never features): report_date,
client_hash_id, content_hash_id.

Excluded: client_has_gsc / client_has_ga4 (these describe the client's
access setup, not page performance) and any FlyRank product score, since
none of those exist in this table but I would exclude them the same way
if they appeared.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Query 1: prove the grain (one row = one page, one client, one day)

In [11]:
con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM {REL}
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,n


Query 2: row count and date span

In [12]:
con.sql(f"""
    SELECT COUNT(*) AS total_rows, MIN(report_date) AS first_day, MAX(report_date) AS last_day
    FROM {REL}
""").df()

,total_rows,first_day,last_day
0,9841378,2026-03-01,2026-03-31


Query 3: availability, using IS TRUE

In [13]:
con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows
    FROM {REL}
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows,ga4_available_rows
0,9841378,3611061.0,413966.0


Grain check: querying report_date + client_hash_id + content_hash_id and
counting duplicates returned zero rows. This confirms one row really is
one page, for one client, on one specific day.

Row count and window: March 2026 contains 9,841,378 rows, spanning the
full month from March 1 to March 31, 2026.

Availability: out of 9,841,378 rows, only 3,611,061 (36.7%) have real GSC
(search) data marked available, and only 413,966 (4.2%) have real GA4
(analytics) data available. This is a big gap. It means most rows in this
table cannot be used for GA4-based signals like sessions or scroll events,
since the ga4_data_available flag is not TRUE for most of them.

In [14]:
mmonth_df = con.sql(f"""
    WITH base AS (
        SELECT
            content_hash_id, report_date,
            gsc_impressions, gsc_clicks, gsc_avg_position, ga4_sessions, scroll_events,
            LAG(gsc_impressions) OVER (PARTITION BY content_hash_id ORDER BY report_date) AS prev_day_impressions,
            LEAD(gsc_impressions) OVER (PARTITION BY content_hash_id ORDER BY report_date) AS next_day_impressions
        FROM {REL}
        WHERE gsc_data_available IS TRUE
    )
    SELECT * FROM base WHERE prev_day_impressions IS NOT NULL
    LIMIT 200000
""").df()

month_df = month_df.dropna(subset=["prev_day_impressions", "next_day_impressions"])

month_df["declines_next_day"] = (month_df["next_day_impressions"] < month_df["prev_day_impressions"] * 0.5).astype(int)
print(month_df["declines_next_day"].value_counts())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

declines_next_day
0    168559
1     21942
Name: count, dtype: int64


In [15]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

honest_features = ["gsc_impressions", "gsc_clicks", "gsc_avg_position", "ga4_sessions", "scroll_events"]
X = month_df[honest_features].fillna(0)
y = month_df["declines_next_day"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
model = DecisionTreeClassifier(max_depth=4, random_state=42)
model.fit(X_train, y_train)
honest_score = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
print(f"Honest AUC using 5 real features: {honest_score:.3f}")

Honest AUC using 5 real features: 0.634


In [16]:
month_df["leaky_next_day_impressions"] = month_df["next_day_impressions"]

leaky_features = honest_features + ["leaky_next_day_impressions"]
X_leak = month_df[leaky_features].fillna(0)

X_train, X_test, y_train, y_test = train_test_split(X_leak, y, test_size=0.3, random_state=42)
model_leak = DecisionTreeClassifier(max_depth=4, random_state=42)
model_leak.fit(X_train, y_train)
leaky_score = roc_auc_score(y_test, model_leak.predict_proba(X_test)[:, 1])
print(f"Leaky AUC (with the cheat column): {leaky_score:.3f}")

Leaky AUC (with the cheat column): 0.770


Five features used (all knowable at decision time):

- gsc_impressions: recorded for that specific day, known as soon as the day ends.
- gsc_clicks: same, recorded per day, known immediately after.
- gsc_avg_position: Google's ranking position that day, already observed by day's end.
- ga4_sessions: analytics sessions for that day, known once the day is logged.
- scroll_events: recorded activity for that day, known the same way.

None of these use any information from a day that hasn't happened yet.

The trap: I added one extra column, leaky_next_day_impressions, which is
literally tomorrow's impression count copied in early. Since my label
(declines_next_day) is defined directly from tomorrow's impressions, this
column basically hands the model the answer.

Honest AUC using only the 5 real features: 0.634
Leaky AUC with the cheat column added: 0.770

The jump from 0.634 to 0.770 shows what leakage looks like: a big,
suspicious improvement that isn't a genuinely better model, just a model
that got to peek at the answer. I removed the leaky column afterward and
kept 0.634 as my honest number.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This slice has real limits worth naming honestly:

- GA4 (analytics) data is only available for 4.2% of rows in March 2026.
  Any feature using ga4_sessions or scroll_events is really only learned
  from a small, possibly unusual group of pages, not the whole inventory.

- This is one month only (March 2026). Content behavior can shift across
  seasons, so a pattern found here may not hold the same way in other months.

- The panel is unbalanced across clients (some have much longer history
  than others), so results may lean toward clients with more data present,
  not necessarily the ones that matter most.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.